# YOLO12s + SPD-Conv — clean Kaggle logs

Salinan dari notebook SPD-Conv asli. Arsitektur, data, bobot pretrained, dan hyperparameter dipertahankan. Perbedaannya hanya pada tampilan log: progress bar per batch dimatikan dan ringkasan loss serta metrik validation dicetak setiap 20 epoch.

In [ ]:
# 1. Clone branch SPD-Conv dan install source training. Aktifkan GPU dan Internet di Kaggle.
import json
import logging
import platform
import subprocess
import sys
import zipfile
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
REPO_BRANCH = 'yolo12-spd-conv'
REPO_DIR = WORKDIR / 'yolo-aceh-rdd2022'

def log_section(title: str) -> None:
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

log_section('CLONE AND INSTALL MODIFIED REPOSITORY')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
REPO_METADATA = WORKDIR / 'repository_revision.txt'
REPO_METADATA.write_text(f'repository={REPO_URL}\nbranch={REPO_BRANCH}\ncommit={REPO_COMMIT}\n', encoding='utf-8')
sys.path.insert(0, str(REPO_DIR))

import torch
import ultralytics
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'PyTorch={torch.__version__} | Ultralytics={ultralytics.__version__} | device={DEVICE} | commit={REPO_COMMIT}')


In [ ]:
# 2. Dataset dan hyperparameter — identik dengan notebook SPD-Conv asli.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd-2022/datasets-china-split-fix')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12-spd.yaml'
CUSTOM_SOURCE_FILES = (
    REPO_DIR / 'ultralytics/nn/modules/conv.py', REPO_DIR / 'ultralytics/nn/modules/__init__.py',
    REPO_DIR / 'ultralytics/nn/tasks.py', MODEL_YAML,
)
EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED, LOG_EVERY = 0, 2, 42, 20
EXPERIMENT_NAME = 'yolo12s_spd_clean_logs_ch_rdd2022_pretrained'
RUNS_DIR = WORKDIR / 'runs'
assert IMGSZ % 32 == 0, 'SPD-Conv requires imgsz divisible by 32.'

DATA_YAML.write_text(f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''', encoding='utf-8')
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'
assert MODEL_YAML.exists() and all(path.exists() for path in CUSTOM_SOURCE_FILES)
print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, nbs={NBS}, optimizer={OPTIMIZER}, lr0={LR0}, seed={SEED}, log_every={LOG_EVERY}')


In [ ]:
# 3. Transfer hanya tensor pretrained YOLO12s yang nama dan bentuknya kompatibel.
from ultralytics import YOLO
PRETRAINED_WEIGHTS = 'yolo12s.pt'

log_section('PRETRAINED WEIGHT TRANSFER')
model = YOLO(str(MODEL_YAML))
source_model = YOLO(PRETRAINED_WEIGHTS).model.float()
target_state = model.model.state_dict()
transferred_state = {
    key: value for key, value in source_model.state_dict().items()
    if key in target_state and target_state[key].shape == value.shape
}
incompatible = model.model.load_state_dict(transferred_state, strict=False)
PRETRAINED_REPORT = {
    'source_weights': PRETRAINED_WEIGHTS,
    'policy': 'same layer index and tensor shape only; SPD-Conv and 5-class head tensors remain trainable',
    'transferred_tensors': len(transferred_state), 'target_tensors': len(target_state),
    'uninitialized_tensors': len(incompatible.missing_keys),
}
model.ckpt = {'model': model.model}
del source_model, target_state, transferred_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(json.dumps(PRETRAINED_REPORT, indent=2))


In [ ]:
# 4. Training clean-log: bar per batch dimatikan; ringkasan tampil setiap 20 epoch.
from ultralytics.utils import LOGGER
import ultralytics.engine.trainer as trainer_module
import ultralytics.engine.validator as validator_module

EPOCH_SUMMARIES = []
ORIGINAL_LOG_LEVEL = LOGGER.level
ORIGINAL_TRAIN_TQDM = trainer_module.TQDM
ORIGINAL_VALIDATOR_TQDM = validator_module.TQDM

def quiet_tqdm(*args, **kwargs):
    kwargs['disable'] = True
    return ORIGINAL_TRAIN_TQDM(*args, **kwargs)

def as_float(value):
    return float(value.detach().cpu()) if isinstance(value, torch.Tensor) else float(value)

def first_metric(metrics, *keys):
    for key in keys:
        if key in metrics:
            return as_float(metrics[key])
    return None

def print_epoch_summary(trainer):
    epoch = trainer.epoch + 1
    if epoch % LOG_EVERY and epoch != trainer.epochs and not trainer.stop:
        return
    metrics = trainer.metrics or {}
    summary = {
        'epoch': f'{epoch}/{trainer.epochs}',
        'box_loss': as_float(trainer.tloss.get('box_loss', 0.0)),
        'cls_loss': as_float(trainer.tloss.get('cls_loss', 0.0)),
        'dfl_loss': as_float(trainer.tloss.get('dfl_loss', 0.0)),
        'precision': first_metric(metrics, 'metrics/precision(B)'),
        'recall': first_metric(metrics, 'metrics/recall(B)'),
        'map50': first_metric(metrics, 'metrics/mAP50(B)'),
        'map50_95': first_metric(metrics, 'metrics/mAP50-95(B)'),
    }
    EPOCH_SUMMARIES.append(summary)
    print('\n' + '=' * 88)
    print('EPOCH SUMMARY')
    print(json.dumps(summary, indent=2))
    print('=' * 88)

trainer_module.TQDM = quiet_tqdm
validator_module.TQDM = quiet_tqdm
model.add_callback('on_fit_epoch_end', print_epoch_summary)

log_section('TRAINING — YOLO12S + SPD-CONV')
LOGGER.setLevel(logging.WARNING)
try:
    model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS, device=DEVICE, workers=WORKERS,
        project=str(RUNS_DIR), name=EXPERIMENT_NAME, exist_ok=True, pretrained=True, optimizer=OPTIMIZER,
        lr0=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, cos_lr=False, patience=PATIENCE,
        seed=SEED, plots=True, verbose=False,
    )
finally:
    LOGGER.setLevel(ORIGINAL_LOG_LEVEL)

RUN_DIR, BEST_PT, LAST_PT = Path(model.trainer.save_dir), Path(model.trainer.best), Path(model.trainer.last)
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')


In [ ]:
# 5. Evaluasi best.pt, simpan riwayat ringkasan 20 epoch, lalu buat ZIP.
log_section('BEST CHECKPOINT EVALUATION')
best_model = YOLO(str(BEST_PT))

def metric_summary(metrics) -> dict:
    return {
        'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
        'save_dir': str(metrics.save_dir),
    }

val_metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                             project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_val', exist_ok=True, plots=True)
EVALUATION_REPORT = {'validation': metric_summary(val_metrics)}
test_labels = DATA_ROOT / 'test' / 'labels'
if test_labels.exists() and any(test_labels.glob('*.txt')):
    test_metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                                  project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_test', exist_ok=True, plots=True)
    EVALUATION_REPORT['test'] = metric_summary(test_metrics)
    TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
else:
    predictions = best_model.predict(source=str(DATA_ROOT / 'test' / 'images'), imgsz=IMGSZ, device=DEVICE,
                                    conf=0.25, save=True, save_txt=True, project=str(RUNS_DIR),
                                    name=f'{EXPERIMENT_NAME}_test_predictions', exist_ok=True)
    TEST_OUTPUT_DIR = Path(predictions[0].save_dir) if predictions else RUNS_DIR
    EVALUATION_REPORT['test'] = {'status': 'labels unavailable; prediction only', 'save_dir': str(TEST_OUTPUT_DIR)}

EPOCH_SUMMARIES_JSON = WORKDIR / f'{EXPERIMENT_NAME}_epoch_summaries.json'
EPOCH_SUMMARIES_JSON.write_text(json.dumps(EPOCH_SUMMARIES, indent=2), encoding='utf-8')
EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(json.dumps({
    'dataset_root': str(DATA_ROOT), 'repository_url': REPO_URL, 'repository_branch': REPO_BRANCH,
    'repository_commit': REPO_COMMIT, 'model_yaml': str(MODEL_YAML),
    'log_every_epochs': LOG_EVERY, 'epoch_summaries_file': str(EPOCH_SUMMARIES_JSON),
    'spd_conv_positions': 'P3/8, P4/16, P5/32', 'pretrained_transfer': PRETRAINED_REPORT,
    'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'nbs': NBS, 'optimizer': OPTIMIZER,
    'lr0': LR0, 'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
}, indent=2), encoding='utf-8')

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
def add_to_zip(archive: zipfile.ZipFile, path: Path) -> int:
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    count = sum(add_to_zip(archive, Path(item)) for item in (
        RUN_DIR, Path(val_metrics.save_dir), TEST_OUTPUT_DIR, DATA_YAML, *CUSTOM_SOURCE_FILES,
        REPO_METADATA, RUN_CONFIG, EVALUATION_JSON, EPOCH_SUMMARIES_JSON,
    ))
print(json.dumps(EVALUATION_REPORT, indent=2))
print(f'ZIP created : {ZIP_PATH} ({count} files)')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))
